# 失败邻域 × 准确读数对照 × 双向局部响应（96 GB v1）

接在 `AxisDose_FailureFamilies_SignedTransport_CPU_v1` 之后。

[实验设置与运行说明](../../../docs/experiment_notes/07_appendix_lora_training/0be3782684f08280.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(total=total, initial=initial, desc=label, unit='cell',
               dynamic_ncols=True, leave=True)
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 16 tracked cells.')

In [ ]:
%%cell_progress 1/16 Mount Drive and cache toolkit
cell_stage('Mount Drive and cache toolkit')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Auditing candidate project roots')
from pathlib import Path
import shutil

PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
REQUIRED_TOOLKIT_PATHS = (
    'pyproject.toml',
    'src/visual_torque_toolkit/__init__.py',
    'src/visual_torque_toolkit/contracts.py',
    'src/visual_torque_toolkit/classify.py',
    'src/visual_torque_toolkit/dataset.py',
    'src/visual_torque_toolkit/edits.py',
    'src/visual_torque_toolkit/generate.py',
    'src/visual_torque_toolkit/prompts.py',
    'src/visual_torque_toolkit/render.py',
    'src/visual_torque_toolkit/splits.py',
    'src/visual_torque_toolkit/world.py',
    'src/visual_torque_toolkit/behavior/__init__.py',
    'src/visual_torque_toolkit/behavior/adapter.py',
    'src/visual_torque_toolkit/behavior/data.py',
    'src/visual_torque_toolkit/behavior/schema.py',
)

def missing_toolkit_paths(project_root):
    toolkit_root = project_root / 'toolkit'
    return [
        relative for relative in REQUIRED_TOOLKIT_PATHS
        if not (toolkit_root / relative).is_file()
    ]

candidate_audit = {
    str(candidate): missing_toolkit_paths(candidate)
    for candidate in PROJECT_CANDIDATES
}
complete_candidates = [
    candidate for candidate in PROJECT_CANDIDATES
    if not candidate_audit[str(candidate)]
]
assert complete_candidates, (
    'No complete VisualTorque toolkit was found. '
    f'Missing-file audit: {candidate_audit}'
)
PROJECT_ROOT = complete_candidates[0]

cell_stage('Copying immutable toolkit source to local SSD')
DRIVE_TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
TOOLKIT_ROOT = Path('/content/visualtorque_local_response_runtime')
import tempfile
TOOLKIT_ROOT = Path(tempfile.mkdtemp(prefix='visualtorque_local_response_', dir='/content'))
shutil.copytree(DRIVE_TOOLKIT_ROOT, TOOLKIT_ROOT, dirs_exist_ok=True)
print('Project:', PROJECT_ROOT)
print('Toolkit runtime:', TOOLKIT_ROOT)

In [ ]:
%%cell_progress 2/16 Install pinned inference and precision policy
cell_stage('Install pinned inference and precision policy')
import importlib
import importlib.metadata
import json
import os
import subprocess
import sys

cell_stage('Configuring caches before importing Torch')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('HF_HOME', '/content/hf_cache')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

cell_stage('Removing incompatible optional Torch packages')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio', 'torchao'],
    check=False,
)
importlib.invalidate_caches()

cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    '--upgrade-strategy', 'only-if-needed',
    'transformers==5.14.0', 'peft==0.20.0', 'accelerate>=1.10',
    'sentencepiece', 'protobuf', 'huggingface_hub>=0.34',
    'safetensors', 'tqdm>=4.66', 'scikit-learn>=1.5',
    'scipy>=1.13', 'pandas>=2.2', 'matplotlib>=3.8',
    'joblib>=1.4', 'pillow>=10.4',
])

cell_stage('Importing the immutable toolkit copy')
SRC_ROOT = TOOLKIT_ROOT / 'src'
PACKAGE_ROOT = SRC_ROOT / 'visual_torque_toolkit'
source_text = str(SRC_ROOT)
sys.path[:] = [entry for entry in sys.path if entry != source_text]
sys.path.insert(0, source_text)
for module_name in tuple(sys.modules):
    if module_name == 'visual_torque_toolkit' or module_name.startswith('visual_torque_toolkit.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import visual_torque_toolkit
from IPython.display import display
from peft import PeftModel
from scipy.stats import binomtest
from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter
from visual_torque_toolkit.generate import (
    GenerationConfig, generate_multi_stack_scenes,
)
from visual_torque_toolkit.prompts import QuestionKind, generate_questions
from visual_torque_toolkit.render import RenderStyle, render_scene

# Defensive Qwen3.5 compatibility patch for exact candidate scoring.
def append_text_prefix_aligned(self, inputs, prefix):
    if not prefix:
        return inputs
    batch_size = inputs['input_ids'].shape[0]
    prefix_tensor = torch.tensor(
        prefix,
        dtype=inputs['input_ids'].dtype,
        device=inputs['input_ids'].device,
    ).unsqueeze(0).expand(batch_size, -1)
    inputs['input_ids'] = torch.cat((inputs['input_ids'], prefix_tensor), dim=1)
    attention_extension = torch.ones(
        (batch_size, len(prefix)),
        dtype=inputs['attention_mask'].dtype,
        device=inputs['attention_mask'].device,
    )
    inputs['attention_mask'] = torch.cat(
        (inputs['attention_mask'], attention_extension), dim=1
    )
    for key in ('token_type_ids', 'mm_token_type_ids'):
        if key not in inputs:
            continue
        extension = torch.zeros(
            (batch_size, len(prefix)),
            dtype=inputs[key].dtype,
            device=inputs[key].device,
        )
        inputs[key] = torch.cat((inputs[key], extension), dim=1)
    inputs.pop('position_ids', None)
    return inputs

HuggingFaceMultimodalAdapter._append_text_prefix = append_text_prefix_aligned

NATIVE_NUMERICS_POLICY = {
    'version': 'native-fp32-matmul-1.0',
    'float32_matmul_precision': 'highest',
    'tf32_matmul': False,
}

def native_precision_state():
    return {'float32_matmul_precision': torch.get_float32_matmul_precision(),
            'tf32_matmul': bool(torch.backends.cuda.matmul.allow_tf32)}

def assert_native_precision():
    expected = {key: NATIVE_NUMERICS_POLICY[key] for key in native_precision_state()}
    actual = native_precision_state()
    if actual != expected:
        raise RuntimeError(
            f'Native numerical policy changed: expected {expected}, got {actual}. '
            'Restart and Run all in this local-response notebook; do not reuse v2 captures '
            'or enable TF32 during a frozen comparison.'
        )
    return actual

# Explicitly undo flags left by a previous notebook in a reused Colab process.
torch.set_float32_matmul_precision('highest')
torch.backends.cuda.matmul.allow_tf32 = False
print('Frozen Native precision:', assert_native_precision())

toolkit_path = Path(visual_torque_toolkit.__file__).resolve()
assert PACKAGE_ROOT.resolve() in toolkit_path.parents
try:
    torchao_version = importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError:
    torchao_version = None
assert torchao_version is None
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('transformers:', importlib.metadata.version('transformers'))
print('peft:', importlib.metadata.version('peft'))
print('Toolkit:', toolkit_path)

In [ ]:
%%cell_progress 3/16 Verify complete source experiment and adapter
cell_stage('Verify complete source experiment and adapter')
import copy, gc, hashlib, itertools, math, tempfile, time
from collections import Counter, defaultdict
from contextlib import contextmanager
from datetime import datetime, timezone
from zipfile import ZipFile
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from IPython.display import Markdown

IMPLEMENTATION_SHA256='sha256:9d98be20b7b0fba189c716b98d88519f774d2e94c89c3dfe86b3960bff484706'
RUN_VERSION='v1'
SEED=20260907
EXPANSION_SEED=20260907
MODEL_ID='Qwen/Qwen3.5-27B'
MODEL_REVISION='373612cb4f9288fcf550b526d86d85febedd9801'
MODEL_DTYPE='bfloat16'
EXPECTED_LAYER_COUNT=64
CAPTURE_LAYERS=(49,55)
MODES=('native','lora')
LABELS=('A','B','C')
MAPPING_VARIANTS=('forward','reverse')
INPUT_FORMS=('image_only','image_plus_facts','facts_only')
DOSES=(-.5,-.25,.25,.5)
PRIMARY_DOSE=.25
PER_CENTER_CELL=4
MAX_NEIGHBOR_L1=3
BOOTSTRAP_REPLICATES=10000
SIGNFLIP_REPLICATES=65536
SMOKE_TOLERANCE=.15
ANCHORS=[
    {'center_id':'near_6x1_1x4','source_family':'axisdose_5c0b824288a2190c21','values':(6,1,1,4)},
    {'center_id':'near_7x3_3x6','source_family':'axisdose_67ff526074719f300f','values':(7,3,3,6)},
    {'center_id':'near_7x3_4x4','source_family':'axisdose_2b1522f9675b9ee2cf','values':(7,3,4,4)},
]
N_FAMILIES=len(ANCHORS)*2*PER_CENTER_CELL
SOURCE_ROOT_OVERRIDE=''
SOURCE_ROOT=(Path(SOURCE_ROOT_OVERRIDE).expanduser().resolve() if SOURCE_ROOT_OVERRIDE else
    PROJECT_ROOT/'results/qwen27b_handoff_axis_dose_crossover_newgeometry_v1')
TRAIN_ROOT=PROJECT_ROOT/'results/qwen27b_bf16_lora_weight_shortcut_v1'
ADAPTER_DIR=TRAIN_ROOT/'adapter_final'
OUTPUT_ROOT=PROJECT_ROOT/'results'/f'qwen27b_local_geometry_oracle_bidirectional_{RUN_VERSION}'
ANALYSIS_DIR=OUTPUT_ROOT/'analysis'
FIGURE_DIR=ANALYSIS_DIR/'figures'
CLEAN_DIR=OUTPUT_ROOT/'clean_cache'
BEHAVIOR_DIR=OUTPUT_ROOT/'behavior_shards'
SHARD_DIR=OUTPUT_ROOT/'intervention_shards'
IMAGE_DIR=OUTPUT_ROOT/'selected_images'
RUNTIME_ROOT=Path(tempfile.mkdtemp(prefix='visualtorque_local_response_',dir='/content'))
for path in (OUTPUT_ROOT,ANALYSIS_DIR,FIGURE_DIR,CLEAN_DIR,BEHAVIOR_DIR,SHARD_DIR,IMAGE_DIR):path.mkdir(parents=True,exist_ok=True)
def canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False,
                      default=lambda x: x.item() if isinstance(x, np.generic) else str(x) if isinstance(x, Path) else _unsupported(x))

def _unsupported(x): raise TypeError(type(x).__name__)

def digest_object(value): return 'sha256:' + hashlib.sha256(canonical_json(value).encode()).hexdigest()

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return 'sha256:' + h.hexdigest()

def read_json(path): return json.loads(Path(path).read_text('utf-8'))

def read_jsonl(path): return [json.loads(s) for s in Path(path).read_text('utf-8').splitlines() if s.strip()]

def write_json(path, data):
    path = Path(path); tmp = path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(canonical_json(data)+'\n', encoding='utf-8'); tmp.replace(path)

def write_jsonl(path, data):
    path = Path(path); tmp = path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(''.join(canonical_json(x)+'\n' for x in data), encoding='utf-8'); tmp.replace(path)

def hash_tree(path):
    return {p.relative_to(path).as_posix():sha256_file(p) for p in sorted(Path(path).rglob('*')) if p.is_file()}

def freeze_json(path, identity):
    identity = json.loads(canonical_json(identity))
    if Path(path).exists():
        assert read_json(path)['identity'] == identity, f'Inputs changed: {path}; use a new RUN_VERSION, never mix runs.'
    else: write_json(path, {'identity':identity, 'locked_at_utc':datetime.now(timezone.utc).isoformat()})
    return digest_object(identity)

def stable_key(value, seed=SEED): return hashlib.sha256(f'{seed}:{value!r}'.encode()).hexdigest()
def require(condition,message):
    if not bool(condition):raise ValueError(message)

def safe_source(relative):
    p=(SOURCE_ROOT/relative).resolve()
    require(p.is_relative_to(SOURCE_ROOT.resolve()),'Invalid parent relative path: '+relative)
    return p

required=['RUN_COMPLETE.json','frozen_protocol.json','runtime_lock.json','geometry_selection.json','scene_manifest.json',
    'prompt_inventory.json','frozen_crossover_axes.pt','capture_identity.json','clean_inventory.json','run_identity.json',
    'source_shard_hashes.json','analysis/clean_records.csv','native_axis_calibration.json','disabled_base_parity.json',
    'historical_native_replay.json','technical_smoke.json','analysis/final_integrity.json']
missing=[str(safe_source(n)) for n in required if not safe_source(n).is_file()]
missing += [str(TRAIN_ROOT/'ADAPTER_MANIFEST.json')] if not (TRAIN_ROOT/'ADAPTER_MANIFEST.json').is_file() else []
require(not missing,'Missing prerequisites (the complete GPU source directory is required):\n'+'\n'.join(missing))
require(torch.cuda.is_available(),'Select a 96 GB GPU session.')
require(torch.cuda.get_device_properties(0).total_memory/2**30>=88,'Unquantized BF16 requires a 96 GB GPU.')
require(importlib.metadata.version('transformers')=='5.14.0' and importlib.metadata.version('peft')=='0.20.0','Restart with the pinned inference packages.')
source_complete=read_json(SOURCE_ROOT/'RUN_COMPLETE.json')
require(source_complete['status']=='complete','The preceding GPU run is not complete.')
for name,expected in tqdm(source_complete['outputs'].items(),desc='Verify preceding GPU artifacts',unit='file'):
    p=safe_source(name);require(p.is_file() and sha256_file(p)==expected,'Parent artifact missing/changed: '+name)
source_protocol=read_json(SOURCE_ROOT/'frozen_protocol.json')['identity']
source_runtime=read_json(SOURCE_ROOT/'runtime_lock.json')['identity']
source_geometry=read_json(SOURCE_ROOT/'geometry_selection.json')['identity']
source_scenes=read_json(SOURCE_ROOT/'scene_manifest.json')['identity']
source_prompts=read_json(SOURCE_ROOT/'prompt_inventory.json')['identity']
source_clean_inventory=read_json(SOURCE_ROOT/'clean_inventory.json')['identity']
source_shards=read_json(SOURCE_ROOT/'source_shard_hashes.json')
source_clean=pd.read_csv(SOURCE_ROOT/'analysis/clean_records.csv',float_precision='round_trip')
require(source_protocol['model']==MODEL_ID and source_protocol['revision']==MODEL_REVISION and source_protocol['dtype']==MODEL_DTYPE,'Wrong source checkpoint.')
require(source_runtime['float32_matmul_precision']=='highest' and source_runtime['tf32_matmul'] is False,'Wrong source numerical policy.')
require(read_json(SOURCE_ROOT/'run_identity.json')['identity']['digest']==source_complete['run_identity'],'Parent run identity mismatch.')
source_axis_hash=sha256_file(SOURCE_ROOT/'frozen_crossover_axes.pt')
source_capture=digest_object({'protocol':digest_object(source_protocol),'runtime':digest_object(source_runtime),'axes':source_axis_hash})
require(read_json(SOURCE_ROOT/'capture_identity.json')['identity']['digest']==source_capture,'Parent capture identity mismatch.')
for name in ['native_axis_calibration.json','disabled_base_parity.json','historical_native_replay.json','technical_smoke.json','analysis/final_integrity.json']:
    require(read_json(SOURCE_ROOT/name)['passed'],'Parent technical gate failed: '+name)
adapter_manifest=read_json(TRAIN_ROOT/'ADAPTER_MANIFEST.json')
require(adapter_manifest['base_model_id']==MODEL_ID and adapter_manifest['base_model_revision']==MODEL_REVISION,'Adapter checkpoint mismatch.')
require(sha256_file(TRAIN_ROOT/'ADAPTER_MANIFEST.json')==source_protocol['source_hashes']['adapter_manifest'],'Adapter manifest differs from preceding run.')
require(hash_tree(ADAPTER_DIR)==adapter_manifest['adapter_files'],'Adapter files changed.')
require(hash_tree(DRIVE_TOOLKIT_ROOT/'src')==source_protocol['toolkit_files'],'Toolkit source differs from preceding run.')
STYLE_SPEC=copy.deepcopy(source_protocol['style'])
# JSON stores RGB tuples as lists; Pillow requires tuples when rendering the frozen style.
for color_field in ('background','ink','block_fill'):STYLE_SPEC[color_field]=tuple(STYLE_SPEC[color_field])
SOURCE_PATHS={'source_complete':SOURCE_ROOT/'RUN_COMPLETE.json','source_protocol':SOURCE_ROOT/'frozen_protocol.json',
    'source_runtime':SOURCE_ROOT/'runtime_lock.json','source_geometry':SOURCE_ROOT/'geometry_selection.json',
    'source_scenes':SOURCE_ROOT/'scene_manifest.json','source_prompts':SOURCE_ROOT/'prompt_inventory.json',
    'source_axes':SOURCE_ROOT/'frozen_crossover_axes.pt','adapter_manifest':TRAIN_ROOT/'ADAPTER_MANIFEST.json'}
anchor_ids={a['source_family'] for a in ANCHORS}
replay_rows=[dict(r) for r in source_scenes if r['family_id'] in anchor_ids]
require(len(replay_rows)==2*len(ANCHORS),'The three historical failure families are missing from the preceding run.')
source_clean_hashes={r['file']:r['sha256'] for r in source_clean_inventory}
for row in replay_rows:
    p=SOURCE_ROOT/'selected_images'/(row['scene_id']+'.png')
    require(p.is_file() and sha256_file(p)==row['image_sha256'],'Original replay image missing/changed: '+str(p))
    SOURCE_PATHS['replay_image/'+row['scene_id']]=p
    for mode,variant in itertools.product(MODES,MAPPING_VARIANTS):
        name=f"{mode}__{row['scene_id']}__{variant}"
        cp=SOURCE_ROOT/'clean_cache'/(name+'.pt');sp=SOURCE_ROOT/'intervention_shards'/(name+'.json')
        require(cp.is_file() and cp.name in source_clean_hashes and sha256_file(cp)==source_clean_hashes[cp.name],'Historical clean cache missing/changed: '+str(cp))
        require(sp.is_file() and sp.name in source_shards and sha256_file(sp)==source_shards[sp.name],'Historical shard missing/changed: '+str(sp))
        SOURCE_PATHS['replay_clean/'+name]=cp;SOURCE_PATHS['replay_shard/'+name]=sp
SOURCE_SYNTHETIC=bool(source_complete.get('synthetic_only',False))
print('Verified preceding GPU experiment, adapter, frozen axes, and all historical replay inputs.')

In [ ]:
%%cell_progress 4/16 Freeze unseen failure-neighborhood geometry
cell_stage('Freeze unseen failure-neighborhood geometry')
def mirror_key(values):
    u,d,v,e=map(int,values)
    a=(((u,d),),((v,e),))
    return min(a,a[::-1])

def physics_cell(u,d,v,e):
    sign=lambda x:int(x>0)-int(x<0)
    w,q,t=sign(u-v),sign(d-e),sign(u*d-v*e)
    if t and w!=t and q==t:return 'weight_conflict'
    if t and q!=t and w==t:return 'distance_conflict'
    if t and w==t and q==t:return 'both_cues_aligned'
    return 'mixed_nonbalanced'

def neighbor_distance(values,reference):
    u,d,v,e=values
    return min(sum(abs(a-b) for a,b in zip(values,reference)),sum(abs(a-b) for a,b in zip((v,e,u,d),reference)))

def select_neighbors(excluded):
    pools=defaultdict(list)
    for u,d,v,e in itertools.product(range(1,8),range(1,7),range(1,8),range(1,7)):
        if (u,d)>(v,e) or max(u,v) not in (6,7) or u*d==v*e:continue
        values=(u,d,v,e);key=mirror_key(values)
        if canonical_json(key) in excluded:continue
        distances=[neighbor_distance(values,a['values']) for a in ANCHORS]
        closest=min(range(len(ANCHORS)),key=lambda i:(distances[i],i))
        if distances[closest]>MAX_NEIGHBOR_L1:continue
        cell=physics_cell(*values)
        if cell not in ('distance_conflict','weight_conflict'):continue
        center=ANCHORS[closest]['center_id']
        pools[(center,cell)].append({'family_id':'local_'+stable_key(('geometry',key))[:18],
            'values':values,'geometry_key':key,'center_id':center,'diagnostic_cell':cell,
            'neighbor_l1':distances[closest],'max_units':max(u,v),'margin_bin':min(abs(u*d-v*e)//4,5)})
    selected=[]
    for a in ANCHORS:
        for cell in ('distance_conflict','weight_conflict'):
            pool=sorted(pools[(a['center_id'],cell)],key=lambda r:(r['neighbor_l1'],stable_key(('select',r['geometry_key']))))
            require(len(pool)>=PER_CENTER_CELL,f'Insufficient unseen neighbors for {a["center_id"]}/{cell}; do not silently expand radius or reduce sample count.')
            selected.extend(pool[:PER_CENTER_CELL])
    require(len(selected)==N_FAMILIES and len({canonical_json(r['geometry_key']) for r in selected})==N_FAMILIES,'Duplicate/missing new family.')
    return selected,[r for key in sorted(pools) for r in pools[key]]

excluded={canonical_json(k) for group in source_geometry['excluded'].values() for k in group}
excluded.update(canonical_json(mirror_key((r['left_units'],r['left_distance'],r['right_units'],r['right_distance']))) for r in source_scenes)
for anchor in ANCHORS:
    rows=[r for r in replay_rows if r['family_id']==anchor['source_family']]
    require(all(mirror_key((r['left_units'],r['left_distance'],r['right_units'],r['right_distance']))==mirror_key(anchor['values']) for r in rows),'Historical failure geometry changed.')
families,eligible=select_neighbors(excluded)
freeze_json(OUTPUT_ROOT/'geometry_selection.json',{'anchors':ANCHORS,'families':families,'eligible':eligible,
    'excluded_mirror_keys':sorted(excluded),'radius':MAX_NEIGHBOR_L1,'per_center_cell':PER_CENTER_CELL,
    'selection':'nearest unique center; distance then fixed hash; no new outcome filtering'})
from visual_torque_toolkit.world import Scene, Side, Stack
scene_manifest=[];image_hashes={}
for family in tqdm(families,desc='Render unseen neighbors',unit='family'):
    u,d,v,e=family['values']
    base=Scene(family['family_id'],(Stack('left',Side.LEFT,d,u),Stack('right',Side.RIGHT,e,v)))
    ap=RUNTIME_ROOT/(family['family_id']+'__anchor.png');mp=RUNTIME_ROOT/(family['family_id']+'__mirror.png')
    render_scene(base,ap,RenderStyle(**STYLE_SPEC))
    with Image.open(ap) as im:ImageOps.mirror(im.convert('RGB')).save(mp)
    for orientation,path,values in [('anchor',ap,(u,d,v,e)),('mirror',mp,(v,e,u,d))]:
        lu,ld,ru,rd=values;sid=family['family_id']+'__'+orientation;torque=lu*ld-ru*rd
        exact='left' if torque>0 else 'right';h=sha256_file(path);persistent=IMAGE_DIR/path.name
        if persistent.is_file():require(sha256_file(persistent)==h,'New image changed on resume.')
        else:shutil.copy2(path,persistent)
        image_hashes[sid]=h
        scene_manifest.append({'family_id':family['family_id'],'scene_id':sid,'pair_instance_id':family['family_id'],
            'center_id':family['center_id'],'neighbor_l1':family['neighbor_l1'],'cohort':'new_neighbors',
            'diagnostic_cell':family['diagnostic_cell'],'max_units':family['max_units'],'margin_bin':family['margin_bin'],
            'orientation':orientation,'left_units':lu,'left_distance':ld,'right_units':ru,'right_distance':rd,
            'net_torque':torque,'exact_semantic':exact,'side':exact,'shortcut_semantic':'right' if exact=='left' else 'left',
            'image_sha256':h,'runtime_image_path':str(path)})
for row in replay_rows:
    anchor=next(a for a in ANCHORS if a['source_family']==row['family_id'])
    p=SOURCE_PATHS['replay_image/'+row['scene_id']];target=RUNTIME_ROOT/p.name;shutil.copy2(p,target)
    persistent=IMAGE_DIR/p.name
    if persistent.is_file():require(sha256_file(persistent)==row['image_sha256'],'Replay image changed.')
    else:shutil.copy2(p,persistent)
    row.update(cohort='old_failure_diagnostic',center_id=anchor['center_id'],neighbor_l1=0,runtime_image_path=str(target))
    image_hashes[row['scene_id']]=row['image_sha256']
all_scenes=scene_manifest+replay_rows
require(len(scene_manifest)==2*N_FAMILIES,'Wrong new image count.')
freeze_json(OUTPUT_ROOT/'scene_manifest.json',[{k:v for k,v in r.items() if k!='runtime_image_path'} for r in all_scenes])
print('New confirmation neighbors:',N_FAMILIES,'| old diagnostic families:',len(ANCHORS))
display(pd.DataFrame(families).groupby(['center_id','diagnostic_cell']).size().rename('families').reset_index())
display(pd.DataFrame(families)[['family_id','center_id','values','diagnostic_cell','neighbor_l1']])

In [ ]:
%%cell_progress 5/16 Freeze input forms doses and four primary tests
cell_stage('Freeze input forms doses and four primary tests')
import math
from dataclasses import replace as dataclass_replace
from visual_torque_toolkit.behavior.data import BehaviorExample

RULE_TEXT = (
    'Each square is one unit of weight. Distance is measured in tick '
    'intervals from the center pivot. Torque equals weight times distance. '
    'Total torque on a side is the sum over stacks on that side.'
)
QUESTION_TEXT = {
    'outcome': 'Which way will the balance scale tip?',
    'weight_more': 'Which side has more total weight?',
    'weight_less': 'Which side has less total weight?',
}
SEMANTIC_TEXT = {'left': 'left side', 'balanced': 'equal', 'right': 'right side'}

def outcome(value):
    return 'left' if value > 0 else 'right' if value < 0 else 'balanced'

def task_semantic(features, task):
    if task == 'outcome':
        value = int(features['net_torque'])
    elif task == 'weight_more':
        value = int(features['weight_score'])
    elif task == 'weight_less':
        value = -int(features['weight_score'])
    else:
        raise KeyError(task)
    return outcome(value)

def mapping_by_label(semantics):
    assert set(semantics) == {'left', 'balanced', 'right'}
    return tuple({
        'position': index, 'label': LABELS[index], 'semantic': semantic,
    } for index, semantic in enumerate(semantics))

def direction_reference_mapping(row):
    semantics = ['left', 'balanced', 'right']
    offset = int(stable_key(
        f"options:{row['family_id']}", seed=DIRECTION_SEED
    )[:8], 16) % 3
    return mapping_by_label(semantics[offset:] + semantics[:offset])

def target_mappings(row):
    exact, wrong = row['exact_semantic'], row['shortcut_semantic']
    assert {exact, wrong} == {'left', 'right'}
    balanced_label = LABELS[int(stable_key(
        ('balanced-label', row['pair_instance_id']), seed=EXPANSION_SEED
    )[:8], 16) % 3]
    side_labels = [label for label in LABELS if label != balanced_label]
    result = {}
    for variant, exact_label, wrong_label in (
        ('forward', side_labels[0], side_labels[1]),
        ('reverse', side_labels[1], side_labels[0]),
    ):
        by_label = {
            exact_label: exact, wrong_label: wrong, balanced_label: 'balanced'
        }
        result[variant] = mapping_by_label(tuple(by_label[label] for label in LABELS))
    return result

def semantics_by_label(mapping):
    return tuple(next(
        item['semantic'] for item in mapping if item['label'] == label
    ) for label in LABELS)

def label_for(mapping, semantic):
    return next(item['label'] for item in mapping if item['semantic'] == semantic)

def custom_prompt(task, mapping):
    options = '\n'.join(
        f"{item['label']}. {SEMANTIC_TEXT[item['semantic']]}" for item in mapping
    )
    return '\n\n'.join([
        'Answer this question using only the supplied balance-scale scene.',
        RULE_TEXT, 'Question: ' + QUESTION_TEXT[task], 'Options:\n' + options,
        'Do not explain your reasoning. Reply with exactly one option letter '
        'and nothing else. Valid replies are: A, B, C.',
    ])

def make_direction_example(row, role_record, task, mapping, name):
    features = role_record['features']
    semantics = semantics_by_label(mapping)
    gold = task_semantic(features, task)
    return BehaviorExample(
        example_id=f"exactphysics_direction:{row['family_id']}:{name}",
        question_id=f"exactphysics_direction_{row['family_id']}_{name}",
        scene_id=str(role_record['scene_id']), condition=PromptCondition.IMAGE_RULE,
        question_kind=task, variant='exactphysics_direction', split='direction_support',
        problem_type='semantic_direction', outcome=gold,
        net_torque=int(features['net_torque']), prompt=custom_prompt(task, mapping),
        image_path=(DIRECTION_RUNTIME_DIR / role_record['image_filename']).resolve(),
        labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
        option_semantics=semantics, gold_label=LABELS[semantics.index(gold)],
        gold_semantic=gold,
    )

def build_direction_examples(row):
    mapping = direction_reference_mapping(row)
    examples = {}
    for task in DONOR_TASKS:
        for role in ('base', 'aligned_source'):
            key = f'{task}__{role}'
            examples[key] = make_direction_example(
                row, row['roles'][role], task, mapping, key
            )
    return examples, mapping

def build_target_examples(row):
    mappings = target_mappings(row)
    base = outcome_example_by_scene[row['scene_id']]
    examples = {}
    for variant, mapping in mappings.items():
        semantics = semantics_by_label(mapping)
        examples[variant] = dataclass_replace(
            base,
            example_id=f"independent_exactphysics_target:{row['scene_id']}:{variant}",
            question_id=f"independent_exactphysics_target_{row['scene_id']}_{variant}",
            variant=f'independent_exactphysics_{variant}', prompt=custom_prompt('outcome', mapping),
            labels=LABELS, option_text=tuple(SEMANTIC_TEXT[x] for x in semantics),
            option_semantics=semantics,
            gold_label=LABELS[semantics.index(row['exact_semantic'])],
            gold_semantic=row['exact_semantic'],
        )
    return examples, mappings

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(output.last_hidden_state):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported output: {type(output).__name__}')
    assert hidden.ndim == 3
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(type(output).__name__)

def prepare_examples(examples):
    examples = tuple(examples)
    plan = adapter._candidate_plan(examples[0])
    if plan is None:
        raise RuntimeError('A/B/C labels must be single-token candidates.')
    for example in examples[1:]:
        assert adapter._candidate_plan(example) == plan
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(adapter._template(
            [example.messages() for example in examples], add_generation_prompt=True,
        )), fixed_prefix,
    )
    return {'examples': examples, 'inputs': inputs, 'token_by_label': token_by_label}

def final_mask(prepared):
    assert len(prepared['examples']) == 1
    attention = prepared['inputs']['attention_mask'][0].detach().cpu().bool()
    mask = torch.zeros_like(attention)
    mask[int(torch.where(attention)[0][-1])] = True
    return mask

print('Prompt builders ready.')

original_target_mappings=target_mappings
def target_mappings(row):
    global EXPANSION_SEED
    saved_seed=EXPANSION_SEED
    try:
        if row.get('cohort')=='old_failure_diagnostic':EXPANSION_SEED=int(source_protocol['mapping_seed'])
        return original_target_mappings(row)
    finally:EXPANSION_SEED=saved_seed

outcome_example_by_scene={}
for row in all_scenes:
    outcome_example_by_scene[row['scene_id']]=BehaviorExample(example_id=row['scene_id'],question_id=row['scene_id'],
        scene_id=row['scene_id'],condition=PromptCondition.IMAGE_RULE,question_kind='outcome',variant='local_response',
        split=row['cohort'],problem_type='single_stack',outcome=row['exact_semantic'],net_torque=int(row['net_torque']),
        prompt='',image_path=Path(row['runtime_image_path']),labels=LABELS,option_text=('left side','equal','right side'),
        option_semantics=('left','balanced','right'),gold_label='A' if row['exact_semantic']=='left' else 'C',gold_semantic=row['exact_semantic'])

def readout_text(row):
    return (f"Accurate scene readout: the left stack has {row['left_units']} unit squares at distance {row['left_distance']} tick intervals from the pivot. "
        f"The right stack has {row['right_units']} unit squares at distance {row['right_distance']} tick intervals from the pivot.")

def behavior_example(row,variant,form):
    e=build_target_examples(row)[0][variant]
    if form=='image_only':return e
    facts=readout_text(row)
    prompt=e.prompt.replace('Question: ',facts+'\n\nQuestion: ',1)
    if form=='image_plus_facts':return dataclass_replace(e,prompt=prompt,example_id=e.example_id+'__'+form)
    require(form=='facts_only','Unknown input form.')
    prompt=prompt.replace('using only the supplied balance-scale scene.','using only the supplied balance-scale description.')
    return dataclass_replace(e,prompt=prompt,example_id=e.example_id+'__'+form,image_path=None,condition=PromptCondition.SYMBOLIC_RULE)

prompt_inventory=[]
source_prompt_lookup={(r['scene_id'],r['mapping']):r for r in source_prompts}
for row in all_scenes:
    for variant,form in itertools.product(MAPPING_VARIANTS,INPUT_FORMS):
        e=behavior_example(row,variant,form)
        messages=e.messages();has_image=any(x['type']=='image' for m in messages for x in m['content'])
        require(has_image==(form!='facts_only'),'Text-only condition contains an image or image condition lost one.')
        if row['cohort']=='old_failure_diagnostic' and form=='image_only':
            old=source_prompt_lookup[(row['scene_id'],variant)]
            require(e.prompt==old['prompt'] and list(e.option_semantics)==list(old['semantics']) and list(e.labels)==list(old['labels']),'Historical prompt/mapping reproduction failed.')
        prompt_inventory.append({'scene_id':row['scene_id'],'family_id':row['family_id'],'cohort':row['cohort'],
            'mapping':variant,'form':form,'prompt':e.prompt,'labels':list(e.labels),'semantics':list(e.option_semantics),
            'image_sha256':row['image_sha256'] if has_image else None})
PROMPT_DIGEST=freeze_json(OUTPUT_ROOT/'prompt_inventory.json',prompt_inventory)
PLAN=[]
for layer in CAPTURE_LAYERS:
    PLAN.append({'kind':'zero','layer':layer,'axis':'none','dose':0.0})
    for axis,dose in itertools.product(('native','lora','random'),DOSES):PLAN.append({'kind':'add','layer':layer,'axis':axis,'dose':dose})
for p in PLAN:p['condition_id']=digest_object(p).split(':')[1][:24]
require(len(PLAN)==26 and len({p['condition_id'] for p in PLAN})==26,'Wrong condition plan.')
freeze_json(OUTPUT_ROOT/'condition_plan.json',PLAN)
PRIMARY_TEST_NAMES=('lora_facts_accuracy_gain','model_x_facts_accuracy_gain','model_x_side_l55_lora_coordinate_slope','model_x_side_output_margin_slope')
protocol={'implementation':IMPLEMENTATION_SHA256,'model':MODEL_ID,'revision':MODEL_REVISION,'dtype':MODEL_DTYPE,
    'source_run':source_complete['run_identity'],'source_axis_hash':source_axis_hash,'source_hashes':{k:sha256_file(p) for k,p in SOURCE_PATHS.items()},
    'toolkit_files':hash_tree(DRIVE_TOOLKIT_ROOT/'src'),'prompts':PROMPT_DIGEST,'images':image_hashes,'style':STYLE_SPEC,
    'geometry':digest_object(families),'n_families':N_FAMILIES,'anchors':ANCHORS,'per_center_cell':PER_CENTER_CELL,
    'neighbor_l1':MAX_NEIGHBOR_L1,'forms':INPUT_FORMS,'plan':PLAN,'primary_tests':PRIMARY_TEST_NAMES,'primary_dose':PRIMARY_DOSE,
    'primary_projection_coordinate':'lora','primary_injection_layer':49,'primary_injection_axis':'native',
    'primary_margin':'gold semantic minus opposite semantic candidate logprob','new_axis_fitting':False,
    'relative_rule':'abs(dose)*historical_native_norm*recipient_clean_norm/paired_native_clean_norm',
    'seed':SEED,'mapping_seed':EXPANSION_SEED,'bootstrap':BOOTSTRAP_REPLICATES,'signflip':SIGNFLIP_REPLICATES,
    'resampling':'paired mirror-equivalent family; bootstrap stratified by fixed selected center',
    'primary_multiplicity':'Holm over four two-sided tests','old_failures_in_primary':False,
    'native_numerics_policy':NATIVE_NUMERICS_POLICY,'causal_scope':'response to additive intervention; not natural mediation'}
PROTOCOL_DIGEST=freeze_json(OUTPUT_ROOT/'frozen_protocol.json',protocol)
print('Protocol frozen before loading the model:',PROTOCOL_DIGEST)
print('Behavior evaluations:',len(all_scenes)*2*2*len(INPUT_FORMS),'| new-image interventions:',N_FAMILIES*8*len(PLAN))

In [ ]:
%%cell_progress 6/16 Load pinned BF16 base
cell_stage('Load pinned BF16 base')
assert_native_precision()
for key,path in SOURCE_PATHS.items(): assert sha256_file(path)==protocol['source_hashes'][key]
assert hash_tree(ADAPTER_DIR)==adapter_manifest['adapter_files']
if 'adapter' in globals():
    del adapter; gc.collect(); torch.cuda.empty_cache()
adapter = HuggingFaceMultimodalAdapter(MODEL_ID,revision=MODEL_REVISION,dtype=MODEL_DTYPE,device_map='cuda:0')
adapter.model.eval()
def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = (
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'model.model.layers',
        'model.layers',
    )
    for path in candidates:
        try:
            layers = resolve_module_path(model, path)
        except AttributeError:
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
    raise RuntimeError('Could not locate decoder layers.')
LAYER_PATH,DECODER_LAYERS=discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS)==EXPECTED_LAYER_COUNT
# A pre-hook on the base also protects forwards through PEFT's outer wrapper.
# This validates precision; the original scientific intervention hooks are unchanged.
def check_forward_precision(_module, _args):
    assert_native_precision()
_precision_guard_handle = adapter.model.register_forward_pre_hook(check_forward_precision)
linear_attention_implementations=sorted({
    f'{rule.__module__}.{rule.__name__}'
    for module in adapter.model.modules()
    for rule in [getattr(module,'chunk_gated_delta_rule',None)] if rule is not None})
import inspect
model_source_paths={inspect.getsourcefile(type(module)) for module in adapter.model.modules()
                    if 'qwen3_5' in type(module).__module__}
model_source_hashes={Path(path).name:sha256_file(path) for path in model_source_paths if path and Path(path).is_file()}
runtime={'torch':torch.__version__,'transformers':importlib.metadata.version('transformers'),
         'peft':importlib.metadata.version('peft'),'scikit_learn':importlib.metadata.version('scikit-learn'),
         'numpy':np.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),
         'attention_implementation':getattr(adapter.model.config,'_attn_implementation',None),
         'decoder_path':LAYER_PATH,'tf32_matmul':torch.backends.cuda.matmul.allow_tf32,
         'float32_matmul_precision':torch.get_float32_matmul_precision(),
         'native_numerics_policy':NATIVE_NUMERICS_POLICY,
         'linear_attention_implementations':linear_attention_implementations,
         'model_source_hashes':model_source_hashes,
         'cudnn_tf32':bool(torch.backends.cudnn.allow_tf32),
         'nvidia_tf32_override':os.environ.get('NVIDIA_TF32_OVERRIDE'),
         'bf16_reduced_precision_reduction':torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction,
         'pillow':importlib.metadata.version('pillow'),'torchvision':importlib.metadata.version('torchvision')}
RUNTIME_DIGEST=freeze_json(OUTPUT_ROOT/'runtime_lock.json',runtime)
print('BF16 base loaded. Decoder layers:',len(DECODER_LAYERS))
print('Linear attention:',linear_attention_implementations)
print('Precision before inference:',assert_native_precision())

In [ ]:
%%cell_progress 7/16 Verify disabled base and reuse frozen directions
cell_stage('Verify disabled base and reuse frozen directions')
def score_records(prepared, logits):
    logprobs = torch.log_softmax(logits[:, -1, :].float(), dim=-1)
    records = []
    for index, example in enumerate(prepared['examples']):
        scores = [
            float(logprobs[index, prepared['token_by_label'][label]].detach().cpu())
            for label in example.labels
        ]
        maximum = max(scores)
        weights = [math.exp(value - maximum) for value in scores]
        probabilities = [value / sum(weights) for value in weights]
        predicted = int(np.argmax(scores))
        records.append({
            'candidate_labels': list(example.labels),
            'candidate_semantics': list(example.option_semantics),
            'candidate_logprobs': scores,
            'candidate_probabilities': probabilities,
            'predicted_label': example.labels[predicted],
            'predicted_semantic': example.option_semantics[predicted],
            'gold_label': example.gold_label,
            'gold_semantic': example.gold_semantic,
        })
    return records

def selected_norm(tensor):
    return float(torch.linalg.vector_norm(tensor.float()))

def matched_random(tensor, key):
    generator = torch.Generator(device='cpu')
    generator.manual_seed(int(stable_key(key)[:16], 16) % (2**63 - 1))
    noise = torch.randn(tensor.shape, generator=generator, dtype=torch.float32)
    return (noise * (selected_norm(tensor) / selected_norm(noise))).to(tensor.dtype)

def centered_scores(record):
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return values - values.mean()

def cosine_similarity(left, right):
    left = left.float().reshape(-1)
    right = right.float().reshape(-1)
    denominator = torch.linalg.vector_norm(left) * torch.linalg.vector_norm(right)
    assert float(denominator) > 0
    return float(torch.dot(left, right) / denominator)

def rescale_direction(tensor, target_norm):
    norm = selected_norm(tensor)
    assert norm > 0 and target_norm > 0
    return (tensor.float() * (float(target_norm) / norm)).to(tensor.dtype)

def unit_direction(tensor):
    return rescale_direction(tensor, 1.0).float()

def semantic_margin(record, exact_semantic, wrong_semantic):
    semantics = list(record['candidate_semantics'])
    values = np.asarray(record['candidate_logprobs'], dtype=float)
    return float(
        values[semantics.index(exact_semantic)]
        - values[semantics.index(wrong_semantic)]
    )

def vector_projection(delta, direction):
    unit = unit_direction(direction).reshape(-1)
    return float(torch.dot(delta.float().reshape(-1), unit))

def clamp_selected_coordinate(selected, untreated, axis):
    untreated = untreated.to(device=selected.device, dtype=selected.dtype)
    axis_float = axis.to(device=selected.device, dtype=torch.float32)
    axis_float = axis_float / torch.linalg.vector_norm(axis_float)
    displacement = selected.float() - untreated.float()
    coefficient = torch.sum(displacement * axis_float, dim=-1, keepdim=True)
    clamped = selected.float() - coefficient * axis_float
    return clamped.to(selected.dtype), float(coefficient.detach().cpu().item())

def torch_save_atomic(path,payload):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp');torch.save(payload,tmp);tmp.replace(path)

def capture_one(example,layers=CAPTURE_LAYERS):
    assert_native_precision()
    prepared=prepare_examples([example]); mask=final_mask(prepared)
    require(int(mask.sum())==1,'Expected exactly one final token.')
    selected={};handles=[]
    for layer in layers:
        def hook(_module,_inputs,output,layer=layer):
            h=output_hidden_tensor(output)
            selected[layer]=h[0,mask.to(h.device)].detach().cpu().clone()
        handles.append(DECODER_LAYERS[layer].register_forward_hook(hook))
    try:
        with torch.inference_mode(): result=adapter.model(**prepared['inputs'],use_cache=False)
        score=score_records(prepared,result.logits)[0]
    finally:
        for handle in handles:handle.remove()
    require(set(selected)==set(layers),'Some requested layers did not execute.')
    require(all(t.ndim==2 and t.shape[0]==1 and torch.isfinite(t).all() for t in selected.values()),'Invalid final state.')
    return score,selected

def capture_parity_example(example):
    score,selected=capture_one(example)
    return score,{k:v.float().numpy().copy() for k,v in selected.items()}

def compare_disabled_capture(bare_score, disabled_score, bare_selected, disabled_selected):
    score_error=float(np.max(np.abs(centered_scores(disabled_score)-centered_scores(bare_score))))
    assert set(bare_selected)==set(disabled_selected)==set(CAPTURE_LAYERS)
    layers={}
    for layer in CAPTURE_LAYERS:
        x,y=np.asarray(bare_selected[layer]),np.asarray(disabled_selected[layer])
        assert x.shape==y.shape and x.ndim==2 and x.shape[0]==1
        assert np.isfinite(x).all() and np.isfinite(y).all(), 'Nonfinite disabled-base hidden state.'
        layers[str(layer)]={'exactly_equal':bool(np.array_equal(x,y)),
            'max_absolute_error':float(np.max(np.abs(x-y))),
            'relative_l2_error':float(np.linalg.norm(x-y)/max(float(np.linalg.norm(x)),1e-12))}
    return {'centered_score_error':score_error,'layers':layers,
        'passed':bool(np.isfinite(score_error) and score_error<SMOKE_TOLERANCE and
                      all(value['exactly_equal'] for value in layers.values()))}

def model_mode(mode):
    assert_native_precision()
    if mode=='native':
        with adapter.model.disable_adapter():yield
    elif mode=='lora':yield
    else:raise ValueError(mode)
def candidate_summary(score,row):
    by_semantic=dict(zip(score['candidate_semantics'],score['candidate_logprobs']))
    gold=row['exact_semantic'];others=[float(v) for k,v in by_semantic.items() if k!=gold]
    return {'margin':float(by_semantic[gold]-by_semantic[row['shortcut_semantic']]),
        'margin_best_other':float(by_semantic[gold]-max(others)),
        'correct':bool(score['predicted_semantic']==gold),'predicted_semantic':score['predicted_semantic']}

def validate_scores(score,example):
    require(score['candidate_labels']==list(example.labels) and score['candidate_semantics']==list(example.option_semantics),'Candidate mapping changed.')
    values=np.asarray(score['candidate_logprobs'],dtype=float)
    require(values.shape==(3,) and np.isfinite(values).all(),'Invalid candidate scores.')
    require(score['predicted_semantic']==example.option_semantics[int(np.argmax(values))],'Prediction differs from candidate scores.')

def model_token_identity(example):
    prepared=prepare_examples([example])
    inputs=prepared['inputs']
    return digest_object({'input_ids':inputs['input_ids'].detach().cpu().tolist(),
        'attention_mask':inputs['attention_mask'].detach().cpu().tolist(),'candidates':prepared['token_by_label']})

bare_examples=[behavior_example(r,v,'image_only') for r in replay_rows[:2] for v in MAPPING_VARIANTS]
bare_captures=[capture_one(e) for e in bare_examples]
adapter.model=PeftModel.from_pretrained(adapter.model,str(ADAPTER_DIR),is_trainable=False)
adapter.model.eval()
for p in adapter.model.parameters():p.requires_grad_(False)
require(not getattr(adapter.model,'is_loaded_in_4bit',False) and not getattr(adapter.model,'is_loaded_in_8bit',False),'Quantized model is not allowed.')
require(any(torch.count_nonzero(p).item()>0 for n,p in adapter.model.named_parameters() if 'lora_B' in n),'Adapter contains no trained LoRA B.')
@contextmanager
def model_mode(mode):
    assert_native_precision()
    if mode=='native':
        with adapter.model.disable_adapter():yield
    elif mode=='lora':yield
    else:raise ValueError(mode)

parity=[]
with model_mode('native'):
    for e,(score,states) in zip(bare_examples,bare_captures):
        current,hidden=capture_one(e)
        match=all(torch.equal(hidden[l],states[l]) for l in CAPTURE_LAYERS)
        error=float(np.max(np.abs(centered_scores(current)-centered_scores(score))))
        parity.append({'example':e.example_id,'states_exact':match,'centered_score_error':error,'passed':match and error<=1e-5})
write_json(OUTPUT_ROOT/'disabled_base_parity.json',{'passed':all(r['passed'] for r in parity),'rows':parity})
require(all(r['passed'] for r in parity),'Bare base / disabled adapter parity failed.')
del bare_captures
axes_payload=torch.load(SOURCE_ROOT/'frozen_crossover_axes.pt',map_location='cpu',weights_only=True)
require(set(axes_payload)=={'axes','upstream'},'Unexpected frozen axis format.')
AXES=axes_payload['axes']
require(set(AXES)==set(MODES),'Missing axis source.')
axis_rows=[]
for key in ['semantic',*(a+'>'+b for a,b in itertools.permutations(LABELS,2))]:
    a=AXES['native']['semantic'] if key=='semantic' else AXES['native']['code'][key]
    b=AXES['lora']['semantic'] if key=='semantic' else AXES['lora']['code'][key]
    require(torch.isfinite(a).all() and torch.isfinite(b).all() and a.shape==b.shape and selected_norm(a)>0,'Invalid frozen axis.')
    require(abs(selected_norm(a)-selected_norm(b))/selected_norm(a)<1e-5,'Axis norms are no longer matched.')
    axis_rows.append({'axis':key,'native_norm':selected_norm(a),'lora_norm':selected_norm(b),'signed_cosine':cosine_similarity(a,b)})
axis_path=OUTPUT_ROOT/'frozen_crossover_axes.pt'
if axis_path.is_file():require(sha256_file(axis_path)==source_axis_hash,'Output axes changed on resume.')
else:shutil.copy2(SOURCE_ROOT/'frozen_crossover_axes.pt',axis_path)
AXIS_DIGEST=sha256_file(axis_path)
CAPTURE_DIGEST=digest_object({'protocol':PROTOCOL_DIGEST,'runtime':RUNTIME_DIGEST,'axes':AXIS_DIGEST})
freeze_json(OUTPUT_ROOT/'capture_identity.json',{'digest':CAPTURE_DIGEST})
pd.DataFrame(axis_rows).to_csv(ANALYSIS_DIR/'axis_geometry.csv',index=False)
print('Bare/disabled parity passed; original frozen Native and LoRA axes loaded without refitting.')

In [ ]:
%%cell_progress 8/16 Define signed additive interventions
cell_stage('Define signed additive interventions')
def axis_for(layer,source,row,mapping):
    require(layer in CAPTURE_LAYERS and source in MODES,'Unknown scientific axis.')
    if layer==49:return AXES[source]['semantic']*(1 if row['exact_semantic']=='left' else -1)
    transition=label_for(mapping,row['shortcut_semantic'])+'>'+label_for(mapping,row['exact_semantic'])
    return AXES[source]['code'][transition]

def injection_axis(plan,row,variant,mapping):
    if plan['axis']=='random':
        return matched_random(axis_for(plan['layer'],'native',row,mapping),('local_shared_random',row['scene_id'],variant,plan['layer']))
    return axis_for(plan['layer'],plan['axis'],row,mapping)

def make_payload(plan,row,variant,mapping,own,native):
    layer=plan['layer'];reference=selected_norm(axis_for(layer,'native',row,mapping))
    own_norm,native_norm=selected_norm(own[layer]),selected_norm(native[layer])
    require(min(own_norm,native_norm,reference)>0,'Invalid dose denominator.')
    requested=abs(plan['dose'])*reference*own_norm/native_norm
    axis=axis_for(layer,'native',row,mapping) if plan['kind']=='zero' else injection_axis(plan,row,variant,mapping)
    fp32=axis.float()*(requested/selected_norm(axis))*(1 if plan['dose']>=0 else -1)
    cast=fp32.to(own[layer].dtype);effective=(own[layer]+cast).float()-own[layer].float()
    cast_norm=selected_norm(cast)
    require(requested==0 or abs(cast_norm-requested)/requested<.005,'BF16 payload differs from requested norm.')
    meta={'historical_native_reference_norm':reference,'recipient_clean_norm':own_norm,'paired_native_clean_norm':native_norm,
        'requested_payload_norm':requested,'fp32_payload_norm':selected_norm(fp32),'bf16_payload_norm':cast_norm,
        'bf16_payload_over_clean_norm':cast_norm/own_norm,'effective_added_displacement_norm':selected_norm(effective),
        'actual_signed_injected_projection':vector_projection(effective,axis),
        'injection_cosine_native':cosine_similarity(axis,axis_for(layer,'native',row,mapping)),
        'injection_cosine_lora':cosine_similarity(axis,axis_for(layer,'lora',row,mapping))}
    return cast,meta

def evaluate_addition(example,layer,payload,own,row,mapping):
    assert_native_precision()
    prepared=prepare_examples([example]);mask=final_mask(prepared)
    require(int(mask.sum())==1,'Only one final token may be changed.')
    traces={};handles=[]
    for current_layer in CAPTURE_LAYERS:
        def hook(_module,_inputs,output,current_layer=current_layer):
            hidden=output_hidden_tensor(output);indices=mask.to(hidden.device)
            before=hidden[0,indices];after=before
            if current_layer==layer:after=before+payload.to(device=hidden.device,dtype=hidden.dtype)
            delta=after.float()-own[current_layer].to(after.device).float()
            traces[str(current_layer)]={'displacement_from_own_clean_norm':selected_norm(delta),
                **{'projection_on_'+s:vector_projection(delta,axis_for(current_layer,s,row,mapping).to(delta.device)) for s in MODES}}
            if current_layer!=layer:return None
            traces[str(current_layer)]['actual_local_edit_norm']=selected_norm(after.float()-before.float())
            modified=hidden.clone();modified[0,indices]=after
            return replace_output_hidden(output,modified)
        handles.append(DECODER_LAYERS[current_layer].register_forward_hook(hook))
    try:
        with torch.inference_mode():result=adapter.model(**prepared['inputs'],use_cache=False)
        score=score_records(prepared,result.logits)[0]
    finally:
        for handle in handles:handle.remove()
    validate_scores(score,example)
    require(set(traces)=={str(l) for l in CAPTURE_LAYERS},'Missing trace layer.')
    require(all(math.isfinite(float(v)) for t in traces.values() for v in t.values()),'Nonfinite intervention trace.')
    return score,traces

In [ ]:
%%cell_progress 9/16 Replay old anchors and validate technical controls
cell_stage('Replay old anchors and validate technical controls')
replay_checks=[];replay_capture={}
for mode in MODES:
    with model_mode(mode):
        for row in tqdm(replay_rows,desc=mode+': historical anchors',unit='scene'):
            for variant in MAPPING_VARIANTS:
                e=behavior_example(row,variant,'image_only');mapping=target_mappings(row)[variant]
                score,own=capture_one(e)
                key=f"{mode}__{row['scene_id']}__{variant}"
                historical=torch.load(SOURCE_PATHS['replay_clean/'+key],map_location='cpu',weights_only=True)
                require(historical['identity']['capture']==source_capture,'Wrong historical clean capture.')
                validate_scores(historical['score'],e)
                norm_error=max(selected_norm(own[l].float()-historical['states'][l].float())/selected_norm(historical['states'][l]) for l in CAPTURE_LAYERS)
                score_error=float(np.max(np.abs(centered_scores(score)-centered_scores(historical['score']))))
                native_key=f"native__{row['scene_id']}__{variant}"
                native=torch.load(SOURCE_PATHS['replay_clean/'+native_key],map_location='cpu',weights_only=True)['states']
                plan={'kind':'add','layer':49,'axis':'native','dose':1.0}
                payload,meta=make_payload(plan,row,variant,mapping,own,native)
                added,traces=evaluate_addition(e,49,payload,own,row,mapping)
                shard=read_json(SOURCE_PATHS['replay_shard/'+key])
                require(shard['run_identity']==source_complete['run_identity'] and shard['records_digest']==digest_object(shard['records']),'Historical shard identity mismatch.')
                matches=[r for r in shard['records'] if r['kind']=='add' and r['layer']==49 and r['axis']=='native' and r['scale']=='relative' and r['dose']==1]
                require(len(matches)==1,'Missing unique historical L49 +1 anchor.')
                old=matches[0]
                added_error=float(np.max(np.abs(centered_scores(added)-centered_scores(old['score']))))
                projection_error=max(abs(traces['55']['projection_on_'+s]-old['traces']['55']['projection_on_'+s]) for s in MODES)
                passed=(score_error<=SMOKE_TOLERANCE and norm_error<=.01 and added_error<=SMOKE_TOLERANCE and projection_error<=SMOKE_TOLERANCE
                    and score['predicted_semantic']==historical['score']['predicted_semantic'])
                replay_checks.append({'model':mode,'scene_id':row['scene_id'],'mapping':variant,'clean_score_error':score_error,
                    'state_relative_error':norm_error,'add_score_error':added_error,'projection_error':projection_error,'passed':passed})
                replay_capture[(mode,row['scene_id'],variant)]=(score,own)
write_json(OUTPUT_ROOT/'historical_replay.json',{'passed':all(r['passed'] for r in replay_checks),'rows':replay_checks})
require(all(r['passed'] for r in replay_checks),'Historical replay failed; diagnostics saved. Do not loosen thresholds.')
technical=[]
row=replay_rows[0]
for mode in MODES:
    with model_mode(mode):
        for variant in MAPPING_VARIANTS:
            e=behavior_example(row,variant,'image_only');mapping=target_mappings(row)[variant]
            clean,own=replay_capture[(mode,row['scene_id'],variant)]
            native=replay_capture[('native',row['scene_id'],variant)][1]
            for plan in PLAN:
                payload,meta=make_payload(plan,row,variant,mapping,own,native)
                current,traces=evaluate_addition(e,plan['layer'],payload,own,row,mapping)
                actual=traces[str(plan['layer'])]['actual_local_edit_norm']
                dose_error=abs(actual-meta['effective_added_displacement_norm'])
                score_error=float(np.max(np.abs(centered_scores(current)-centered_scores(clean))))
                passed=math.isclose(actual,meta['effective_added_displacement_norm'],abs_tol=1e-4,rel_tol=1e-5)
                if plan['kind']=='zero':passed=passed and score_error<=SMOKE_TOLERANCE and max(t['displacement_from_own_clean_norm'] for t in traces.values())<=SMOKE_TOLERANCE
                technical.append({'model':mode,'mapping':variant,**plan,'dose_error':dose_error,'centered_score_change':score_error,'passed':passed})
# Prepare one real no-image prompt before spending time on confirmation inference.
text_probe=prepare_examples([behavior_example(row,'forward','facts_only')])
require(not any('pixel_values' in k and v is not None for k,v in text_probe['inputs'].items()),'facts_only was processed with image pixels.')
write_json(OUTPUT_ROOT/'technical_smoke.json',{'passed':all(r['passed'] for r in technical),'rows':technical,'facts_only_contains_pixels':False})
require(all(r['passed'] for r in technical),'Zero/edit controls failed; diagnostics saved.')
print('Historical Native/LoRA replay, zero controls, actual BF16 edits, and no-image preparation passed.')
del replay_capture,text_probe

In [ ]:
%%cell_progress 10/16 Score three input forms and freeze doses
cell_stage('Score three input forms and freeze doses')
def clean_path(mode,row,variant):return CLEAN_DIR/f"{mode}__{row['scene_id']}__{variant}.pt"
def clean_identity(mode,row,variant):return {'capture':CAPTURE_DIGEST,'model':mode,'scene_id':row['scene_id'],'mapping':variant,'form':'image_only'}
def load_clean(mode,row,variant):
    path=clean_path(mode,row,variant);data=torch.load(path,map_location='cpu',weights_only=True)
    require(data['identity']==clean_identity(mode,row,variant),'Clean cache identity changed.')
    require(set(data['states'])==set(CAPTURE_LAYERS),'Incomplete clean states.')
    require(all(t.dtype==torch.bfloat16 and t.shape==AXES['native']['semantic'].shape and torch.isfinite(t).all() and selected_norm(t)>0 for t in data['states'].values()),'Invalid clean state.')
    validate_scores(data['score'],behavior_example(row,variant,'image_only'))
    if 'CLEAN_HASHES' in globals():require(sha256_file(path)==CLEAN_HASHES[path.name],'Frozen clean bytes changed.')
    return data

behavior_records=[];clean_inventory=[]
for mode in MODES:
    with model_mode(mode):
        jobs=list(itertools.product(all_scenes,MAPPING_VARIANTS,INPUT_FORMS));durations=[]
        for index,(row,variant,form) in enumerate(tqdm(jobs,desc=mode+': three input forms',unit='input')):
            start=time.perf_counter();e=behavior_example(row,variant,form)
            path=BEHAVIOR_DIR/f"{mode}__{row['scene_id']}__{variant}__{form}.json"
            expected={'capture':CAPTURE_DIGEST,'model':mode,'scene_id':row['scene_id'],'mapping':variant,'form':form}
            if path.is_file():
                saved=read_json(path);require(saved['identity']==expected and saved['score_digest']==digest_object(saved['score']),'Behavior cache mismatch.')
                validate_scores(saved['score'],e)
            else:
                if form=='image_only':
                    cp=clean_path(mode,row,variant)
                    if not cp.is_file():
                        score,states=capture_one(e)
                        torch_save_atomic(cp,{'identity':clean_identity(mode,row,variant),'score':score,'states':states,'token_identity':model_token_identity(e)})
                    data=load_clean(mode,row,variant);score=data['score'];token_identity=data['token_identity']
                else:
                    prepared=prepare_examples([e])
                    if form=='facts_only':require(not any('pixel_values' in k and v is not None for k,v in prepared['inputs'].items()),'Text-only evaluation contains pixels.')
                    with torch.inference_mode():result=adapter.model(**prepared['inputs'],use_cache=False)
                    score=score_records(prepared,result.logits)[0];token_identity=model_token_identity(e)
                validate_scores(score,e)
                saved={'identity':expected,'score':score,'score_digest':digest_object(score),'token_identity':token_identity}
                write_json(path,saved)
            if form=='image_only':
                data=load_clean(mode,row,variant)
                require(data['score']==saved['score'] and data['token_identity']==saved['token_identity'],'Behavior and clean cache disagree.')
                clean_inventory.append({'model':mode,'scene_id':row['scene_id'],'mapping':variant,'cohort':row['cohort'],
                    'file':clean_path(mode,row,variant).name,'sha256':sha256_file(clean_path(mode,row,variant)),
                    'norms':{str(l):selected_norm(t) for l,t in data['states'].items()}})
            meta={k:row[k] for k in ['family_id','scene_id','center_id','cohort','diagnostic_cell','max_units','side','neighbor_l1']}
            score=saved['score']
            behavior_records.append({**meta,'model':mode,'mapping':variant,'form':form,**candidate_summary(score,row),
                'predicted_label':score['predicted_label'],**{'logprob_'+label:float(v) for label,v in zip(score['candidate_labels'],score['candidate_logprobs'])}})
            durations.append(time.perf_counter()-start)
            if index==7:print(mode,'behavior ETA from first eight inputs:',round(np.mean(durations)*(len(jobs)-8)/60,1),'minutes')
behavior_frame=pd.DataFrame(behavior_records)
require(len(behavior_frame)==len(all_scenes)*2*2*3 and not behavior_frame.duplicated(['model','scene_id','mapping','form']).any(),'Incomplete or duplicate behavior cells.')
behavior_frame.to_csv(ANALYSIS_DIR/'behavior_records.csv',index=False)
CLEAN_HASHES={r['file']:r['sha256'] for r in clean_inventory}
require(len(CLEAN_HASHES)==len(all_scenes)*4,'Wrong clean cache count.')
CLEAN_DIGEST=freeze_json(OUTPUT_ROOT/'clean_inventory.json',clean_inventory)
dose_inventory=[]
for mode,row,variant in tqdm(list(itertools.product(MODES,scene_manifest,MAPPING_VARIANTS)),desc='Freeze all doses',unit='input'):
    own=load_clean(mode,row,variant)['states'];native=load_clean('native',row,variant)['states'];mapping=target_mappings(row)[variant]
    for plan in PLAN:
        _,meta=make_payload(plan,row,variant,mapping,own,native)
        dose_inventory.append({'model':mode,'scene_id':row['scene_id'],'mapping':variant,**plan,**meta})
DOSE_DIGEST=freeze_json(OUTPUT_ROOT/'dose_inventory.json',dose_inventory)
RUN_DIGEST=digest_object({'capture':CAPTURE_DIGEST,'clean':CLEAN_DIGEST,'doses':DOSE_DIGEST,'plan':PLAN})
freeze_json(OUTPUT_ROOT/'run_identity.json',{'digest':RUN_DIGEST})
pd.DataFrame(dose_inventory).to_csv(ANALYSIS_DIR/'prescribed_and_realized_doses.csv',index=False)
print('Behavior complete; dose inventory frozen before confirmation intervention scoring.')

In [ ]:
%%cell_progress 11/16 Define validated resumable runner
cell_stage('Define validated resumable runner')
def validate_partial(saved,mode,row,variant,clean):
    require(saved['run_identity']==RUN_DIGEST and saved['clean_sha256']==CLEAN_HASHES[clean_path(mode,row,variant).name],'Run/clean shard identity changed.')
    require((saved['model'],saved['scene_id'],saved['mapping'])==(mode,row['scene_id'],variant),'Wrong intervention shard input.')
    require(saved['records_digest']==digest_object(saved['records']),'Intervention record digest mismatch.')
    records=saved['records'];require(len(records)<=len(PLAN),'Too many cached conditions.')
    e=behavior_example(row,variant,'image_only');baseline=candidate_summary(clean['score'],row)
    for record,plan in zip(records,PLAN):
        require(all(record[k]==v for k,v in plan.items()),'Incomplete/out-of-order/changed condition cache.')
        validate_scores(record['score'],e)
        now=candidate_summary(record['score'],row)
        require(abs(record['margin_shift']-(now['margin']-baseline['margin']))<1e-9,'Margin shift cannot be reproduced.')
        require(record['patched_tokens']==1 and set(record['traces'])=={'49','55'},'Invalid intervention site/token count.')
        require(all(math.isfinite(float(v)) for t in record['traces'].values() for v in t.values()),'Nonfinite cached trace.')
        actual=record['traces'][str(plan['layer'])]['actual_local_edit_norm']
        require(math.isclose(actual,record['dose_metadata']['effective_added_displacement_norm'],abs_tol=1e-4,rel_tol=1e-5),'Realized edit differs from frozen dose.')

def run_mode(mode):
    jobs=list(itertools.product(scene_manifest,MAPPING_VARIANTS));durations=[]
    with model_mode(mode):
        for index,(row,variant) in enumerate(tqdm(jobs,desc=mode+': local interventions',unit='input')):
            start=time.perf_counter();path=SHARD_DIR/f"{mode}__{row['scene_id']}__{variant}.json"
            clean=load_clean(mode,row,variant);own=clean['states'];native=load_clean('native',row,variant)['states']
            e=behavior_example(row,variant,'image_only');mapping=target_mappings(row)[variant]
            saved=(read_json(path) if path.is_file() else {'run_identity':RUN_DIGEST,'model':mode,'scene_id':row['scene_id'],
                'mapping':variant,'clean_sha256':CLEAN_HASHES[clean_path(mode,row,variant).name],'records':[],'records_digest':digest_object([])})
            validate_partial(saved,mode,row,variant,clean)
            if len(saved['records'])==len(PLAN):continue
            require(model_token_identity(e)==clean['token_identity'],'Prepared input changed since clean capture.')
            baseline=candidate_summary(clean['score'],row)
            for plan in PLAN[len(saved['records']):]:
                payload,meta=make_payload(plan,row,variant,mapping,own,native)
                score,traces=evaluate_addition(e,plan['layer'],payload,own,row,mapping)
                summary=candidate_summary(score,row)
                record={**plan,'score':score,'traces':traces,'dose_metadata':meta,'patched_tokens':1,
                    'margin_shift':summary['margin']-baseline['margin'],
                    'margin_best_other_shift':summary['margin_best_other']-baseline['margin_best_other'],
                    'centered_score_change':float(np.max(np.abs(centered_scores(score)-centered_scores(clean['score']))))}
                if plan['kind']=='zero':
                    require(record['centered_score_change']<=SMOKE_TOLERANCE and max(t['displacement_from_own_clean_norm'] for t in traces.values())<=SMOKE_TOLERANCE,
                        'Zero intervention differs from its own clean baseline; stop and inspect numerical state.')
                saved['records'].append(record);saved['records_digest']=digest_object(saved['records'])
                validate_partial(saved,mode,row,variant,clean);write_json(path,saved)
            durations.append(time.perf_counter()-start)
            if len(durations) in (1,4):print(mode,'ETA after',len(durations),'completed inputs:',round(np.mean(durations)*(len(jobs)-index-1)/3600,2),'hours')

In [ ]:
%%cell_progress 12/16 Run Native local interventions
cell_stage('Run Native local interventions')
run_mode('native')

In [ ]:
%%cell_progress 13/16 Run LoRA local interventions
cell_stage('Run LoRA local interventions')
run_mode('lora')

In [ ]:
%%cell_progress 14/16 Verify complete grids and realized edits
cell_stage('Verify complete grids and realized edits')
patch_rows=[];trace_rows=[];shard_hashes={}
for mode,row,variant in itertools.product(MODES,scene_manifest,MAPPING_VARIANTS):
    path=SHARD_DIR/f"{mode}__{row['scene_id']}__{variant}.json";saved=read_json(path);clean=load_clean(mode,row,variant)
    validate_partial(saved,mode,row,variant,clean);require(len(saved['records'])==len(PLAN),'Finish all intervention stages first.')
    shard_hashes[path.name]=sha256_file(path)
    meta={k:row[k] for k in ['family_id','scene_id','center_id','cohort','side','diagnostic_cell','max_units']}
    meta.update(model=mode,mapping=variant)
    for r in saved['records']:
        score=r['score'];summary=candidate_summary(score,row)
        condition={k:r[k] for k in PLAN[0]}
        patch_rows.append({**meta,**condition,**summary,'clean_margin':candidate_summary(clean['score'],row)['margin'],
            'margin_shift':r['margin_shift'],'margin_best_other_shift':r['margin_best_other_shift'],
            'centered_score_change':r['centered_score_change'],**r['dose_metadata'],
            **{'logprob_'+label:float(v) for label,v in zip(score['candidate_labels'],score['candidate_logprobs'])}})
        for layer,trace in r['traces'].items():trace_rows.append({**meta,**condition,'trace_layer':int(layer),**trace})
patch_frame=pd.DataFrame(patch_rows);trace_frame=pd.DataFrame(trace_rows);dose_frame=pd.DataFrame(dose_inventory)
KEY=['model','scene_id','mapping','condition_id']
require(len(patch_frame)==N_FAMILIES*8*26 and not patch_frame.duplicated(KEY).any(),'Intervention grid incomplete/duplicated.')
require(len(trace_frame)==len(patch_frame)*2 and not trace_frame.duplicated(KEY+['trace_layer']).any(),'Trace grid incomplete/duplicated.')
dose_cols=[k for k in dose_frame if k not in KEY+['kind','layer','axis','dose']]
joined=patch_frame[KEY+dose_cols].merge(dose_frame[KEY+dose_cols],on=KEY,suffixes=('_observed','_planned'),validate='one_to_one',how='outer',indicator=True)
require((joined['_merge']=='both').all(),'Frozen and observed dose keys differ.')
for c in dose_cols:require(np.allclose(joined[c+'_observed'],joined[c+'_planned'],rtol=1e-9,atol=1e-10),'Dose metadata mismatch: '+c)
zero=patch_frame[patch_frame.kind=='zero'];zero_trace=trace_frame[trace_frame.kind=='zero']
control={'zero_max_margin_change':float(zero.margin_shift.abs().max()),'zero_max_score_change':float(zero.centered_score_change.max()),
    'zero_max_state_displacement':float(zero_trace.displacement_from_own_clean_norm.max())}
require(max(control.values())<=SMOKE_TOLERANCE,'Zero controls exceeded tolerance.')
ratio=dose_frame[dose_frame.kind=='add'].pivot(index=['scene_id','mapping','layer','axis','dose'],columns='model',values='bf16_payload_over_clean_norm')
control['relative_payload_max_fractional_mismatch']=float(((ratio.lora-ratio.native)/ratio.native).abs().max())
require(control['relative_payload_max_fractional_mismatch']<=.01,'Cross-model relative dose matching failed.')
for name,frame in [('intervention_records',patch_frame),('signed_state_traces',trace_frame)]:frame.to_csv(ANALYSIS_DIR/(name+'.csv'),index=False)
write_json(OUTPUT_ROOT/'source_shard_hashes.json',shard_hashes)
write_json(ANALYSIS_DIR/'control_checks.json',control)
write_json(ANALYSIS_DIR/'final_integrity.json',{'passed':True,'new_families':N_FAMILIES,'behavior':len(behavior_frame),'interventions':len(patch_frame),'traces':len(trace_frame)})
display(pd.DataFrame([control]))
print('Complete behavior/condition grids and frozen dose identities verified.')

In [ ]:
%%cell_progress 15/16 Estimate paired behavior and bidirectional responses
cell_stage('Estimate paired behavior and bidirectional responses')
def estimate(values,key,with_test=False):
    # Series index is the physical family ID. Fixed-center bootstrap keeps center composition.
    values=pd.Series(values,dtype=float).sort_index();require(len(values)>=2 and np.isfinite(values).all(),'Invalid family effects.')
    require(values.index.is_unique and set(values.index)<=set(family_center),'Missing/duplicate family center mapping.')
    x=values.to_numpy();centers=np.asarray([family_center[f] for f in values.index]);rng=np.random.default_rng(int(stable_key(('stats',key))[:16],16))
    draws=np.zeros(BOOTSTRAP_REPLICATES)
    for center in sorted(set(centers)):
        v=x[centers==center]
        for start in range(0,BOOTSTRAP_REPLICATES,256):
            end=min(start+256,BOOTSTRAP_REPLICATES)
            draws[start:end]+=v[rng.integers(0,len(v),size=(end-start,len(v)))].sum(axis=1)/len(x)
    out={'n_families':len(x),'estimate':float(x.mean()),'ci_low':float(np.quantile(draws,.025)),'ci_high':float(np.quantile(draws,.975))}
    if with_test:
        exceed=0
        for start in range(0,SIGNFLIP_REPLICATES,1024):
            count=min(1024,SIGNFLIP_REPLICATES-start);signs=rng.integers(0,2,size=(count,len(x)))*2-1
            exceed+=int((np.abs((signs*x).mean(1))>=abs(x.mean())-1e-12).sum())
        out['p_two_sided']=(exceed+1)/(SIGNFLIP_REPLICATES+1)
    return out
def holm(pvalues):
    pvalues=np.asarray(pvalues,dtype=float);order=np.argsort(pvalues)
    adjusted=np.empty(len(pvalues));running=0.0
    for i,index in enumerate(order):
        running=max(running,(len(pvalues)-i)*pvalues[index]);adjusted[index]=min(1.0,running)
    return adjusted.tolist()
family_center={r['family_id']:r['center_id'] for r in families}
new_behavior=behavior_frame[behavior_frame.cohort=='new_neighbors'].copy()
family_behavior=new_behavior.groupby(['family_id','model','form']).correct.mean().unstack(['model','form'])
require(len(family_behavior)==N_FAMILIES and not family_behavior.isna().any().any(),'Unpaired behavior data.')
behavior_summary=[]
for scope,value,sub in [('all','all',new_behavior)]+[(field,str(value),g) for field in ['center_id','diagnostic_cell','side','mapping'] for value,g in new_behavior.groupby(field)]:
    for (model,form),g in sub.groupby(['model','form']):
        family_values=g.groupby('family_id').correct.mean()
        behavior_summary.append({'scope':scope,'value':value,'model':model,'form':form,**estimate(family_values,('behavior',scope,value,model,form))})
behavior_summary=pd.DataFrame(behavior_summary)
old_diagnostics=behavior_frame[behavior_frame.cohort=='old_failure_diagnostic'].groupby(['family_id','model','form']).agg(
    correct=('correct','sum'),inputs=('correct','size'),margin=('margin','mean')).reset_index()
# Include exact zero observations in curves; never synthesize them from the cached score.
curves=[]
for keys,sub in patch_frame.groupby(['model','layer','axis','dose']):
    for metric in ['margin_shift','margin_best_other_shift','correct','effective_added_displacement_norm']:
        values=sub.groupby('family_id')[metric].mean()
        curves.append({**dict(zip(['model','layer','axis','dose'],keys)),'metric':metric,**estimate(values,('curve',keys,metric))})
curve_frame=pd.DataFrame(curves)
transport=trace_frame[(trace_frame.kind=='add')&(trace_frame.layer==49)&(trace_frame.trace_layer==55)].merge(
    patch_frame[KEY+['margin_shift','margin_best_other_shift','correct']],on=KEY,validate='one_to_one')
slopes=[]
for keys,sub in transport.groupby(['model','family_id','scene_id','mapping','axis']):
    r=sub.iloc[0];by_dose=sub.set_index('dose');require(set(by_dose.index)==set(DOSES),'Missing signed dose pair.')
    for d in (.25,.5):
        for metric in ['projection_on_native','projection_on_lora','margin_shift','margin_best_other_shift']:
            plus,minus=float(by_dose.loc[d,metric]),float(by_dose.loc[-d,metric])
            slopes.append({**dict(zip(['model','family_id','scene_id','mapping','axis'],keys)),'center_id':r.center_id,
                'side':r.side,'diagnostic_cell':r.diagnostic_cell,'dose_magnitude':d,'metric':metric,
                'positive_response':plus,'negative_response':minus,'central_slope':(plus-minus)/(2*d),
                'even_response':(plus+minus)/2})
slope_frame=pd.DataFrame(slopes)
primary_values={}
lora_gain=family_behavior[('lora','image_plus_facts')]-family_behavior[('lora','image_only')]
native_gain=family_behavior[('native','image_plus_facts')]-family_behavior[('native','image_only')]
primary_values['lora_facts_accuracy_gain']=lora_gain
primary_values['model_x_facts_accuracy_gain']=lora_gain-native_gain
for metric,name in [('projection_on_lora','model_x_side_l55_lora_coordinate_slope'),('margin_shift','model_x_side_output_margin_slope')]:
    sub=slope_frame[(slope_frame.axis=='native')&(slope_frame.dose_magnitude==PRIMARY_DOSE)&(slope_frame.metric==metric)]
    w=sub.groupby(['family_id','model','side']).central_slope.mean().unstack(['model','side'])
    require(not w.isna().any().any(),'Unpaired primary side contrast.')
    primary_values[name]=(w[('lora','right')]-w[('lora','left')])-(w[('native','right')]-w[('native','left')])
require(tuple(primary_values)==tuple(PRIMARY_TEST_NAMES),'Primary test set changed.')
primary=pd.DataFrame([{'test':name,**estimate(v,name,True)} for name,v in primary_values.items()])
primary['p_holm']=holm(primary.p_two_sided)
primary_family=pd.DataFrame([{'test':name,'family_id':fid,'center_id':family_center[fid],'value':float(v)} for name,s in primary_values.items() for fid,v in s.items()])
slope_summary=[]
for scope,value,sub in [('all','all',slope_frame)]+[(field,str(value),g) for field in ['center_id','side','diagnostic_cell'] for value,g in slope_frame.groupby(field)]:
    for keys,g in sub.groupby(['model','axis','dose_magnitude','metric']):
        for response in ['central_slope','even_response']:
            values=g.groupby('family_id')[response].mean()
            slope_summary.append({'scope':scope,'value':value,**dict(zip(['model','axis','dose_magnitude','metric'],keys)),
                'response':response,**estimate(values,('slopes',scope,value,keys,response))})
slope_summary=pd.DataFrame(slope_summary)
quadrants=slope_frame.pivot(index=['model','family_id','scene_id','mapping','axis','dose_magnitude'],columns='metric',values='central_slope').reset_index()
for source in MODES:quadrants['projection_output_opposed_'+source]=(quadrants['projection_on_'+source]*quadrants.margin_shift)<0
# Secondary contrast reports model/side/axis interactions directly, not differences of significance.
secondary=[]
for metric in ['projection_on_native','projection_on_lora','margin_shift']:
    for axis,d in itertools.product(('native','lora','random'),(.25,.5)):
        sub=slope_frame[(slope_frame.metric==metric)&(slope_frame.axis==axis)&(slope_frame.dose_magnitude==d)]
        w=sub.groupby(['family_id','model','side']).central_slope.mean().unstack(['model','side'])
        delta=(w[('lora','right')]-w[('lora','left')])-(w[('native','right')]-w[('native','left')])
        secondary.append({'contrast':'model_x_side','metric':metric,'axis':axis,'dose':d,**estimate(delta,('secondary',metric,axis,d))})
secondary=pd.DataFrame(secondary)
for name,frame in [('behavior_summary',behavior_summary),('old_failure_diagnostics',old_diagnostics),('dose_curves',curve_frame),
    ('input_bidirectional_slopes',slope_frame),('slope_summary',slope_summary),('projection_output_quadrants',quadrants),
    ('primary_four_tests',primary),('primary_family_contrasts',primary_family),('secondary_side_interactions',secondary)]:
    frame.to_csv(ANALYSIS_DIR/(name+'.csv'),index=False)
def show(frame):
    with pd.option_context('display.max_rows',180,'display.max_columns',24,'display.width',200):display(frame.round(6))
show(primary);show(behavior_summary[behavior_summary.scope.isin(['all','diagnostic_cell'])]);show(old_diagnostics)
show(slope_summary[(slope_summary.scope=='side')&(slope_summary.axis=='native')&(slope_summary.dose_magnitude==.25)&(slope_summary.response=='central_slope')])
show(secondary)

In [ ]:
%%cell_progress 16/16 Display report figures and export review bundle
cell_stage('Display report figures and export review bundle')
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':160,'axes.spines.top':False,'axes.spines.right':False})
def finish(fig,name):
    fig.tight_layout();fig.savefig(FIGURE_DIR/(name+'.png'),bbox_inches='tight');display(fig);plt.close(fig)
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,group in zip(axes,['all','distance_conflict','weight_conflict']):
    sub=behavior_summary[(behavior_summary.scope=='all') if group=='all' else ((behavior_summary.scope=='diagnostic_cell')&(behavior_summary.value==group))]
    for i,model in enumerate(MODES):
        vals=sub[sub.model==model].set_index('form').reindex(INPUT_FORMS)
        ax.bar(np.arange(3)+(i-.5)*.34,vals.estimate,.34,label=model)
    ax.set_xticks(range(3),['Image','Image + facts','Facts only'],rotation=20);ax.set_ylim(0,1.05);ax.set_title(group);ax.set_ylabel('Accuracy')
axes[0].legend();finish(fig,'01_behavior_readout_conditions')
fig,axes=plt.subplots(2,2,figsize=(11,7))
for i,side in enumerate(['left','right']):
    for j,(metric,title) in enumerate([('projection_on_lora','L55 LoRA-coordinate projection'),('margin_shift','Gold vs opposite margin change')]):
        ax=axes[i,j]
        for model in MODES:
            for axis,style in [('native','-'),('lora','--'),('random',':')]:
                sub=transport[(transport.side==side)&(transport.model==model)&(transport.axis==axis)]
                values=sub.groupby(['family_id','dose'])[metric].mean().groupby('dose').mean()
                ax.plot(values.index,values.values,style,marker='o',label=model+'/'+axis)
        ax.axhline(0,color='gray',lw=.6);ax.axvline(0,color='gray',lw=.6);ax.set(title=side+' | '+title,xlabel='Nominal relative dose')
axes[0,0].legend(fontsize=7);finish(fig,'02_l49_transport_and_output_by_side')
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,model in zip(axes,MODES):
    for axis in ['native','lora','random']:
        sub=curve_frame[(curve_frame.model==model)&(curve_frame.layer==55)&(curve_frame.axis==axis)&(curve_frame.metric=='margin_shift')].sort_values('dose')
        ax.plot(sub.dose,sub.estimate,marker='o',label=axis)
    ax.axhline(0,color='gray',lw=.6);ax.set(title=model+' | direct L55 injection',xlabel='Nominal relative dose',ylabel='Mean margin change');ax.legend()
finish(fig,'03_direct_l55_response')
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,model in zip(axes,MODES):
    sub=quadrants[(quadrants.model==model)&(quadrants.axis=='native')&(quadrants.dose_magnitude==.25)]
    side_lookup={r['scene_id']:r['side'] for r in scene_manifest}
    for side,marker in [('left','o'),('right','^')]:
        part=sub[sub.scene_id.map(side_lookup)==side]
        ax.scatter(part.projection_on_lora,part.margin_shift,marker=marker,alpha=.65,label=side)
    ax.axhline(0,color='gray',lw=.7);ax.axvline(0,color='gray',lw=.7);ax.set(title=model,xlabel='L55 projection finite-difference response',ylabel='Output margin finite-difference response');ax.legend()
fig.suptitle('Paired input responses | Native L49 axis, +/- 0.25 | points share families')
finish(fig,'04_projection_output_joint_response')

def md_table(frame):
    if frame.empty:return '（无记录）'
    def fmt(x):
        if pd.isna(x):return 'NA'
        return f'{x:.5f}' if isinstance(x,(float,np.floating)) else str(x).replace('|','/')
    return '\n'.join(['| '+' | '.join(frame.columns)+' |','| '+' | '.join(['---']*len(frame.columns))+' |']+
        ['| '+' | '.join(fmt(x) for x in row)+' |' for row in frame.itertuples(index=False,name=None)])
lines=['# VisualTorque：新失败邻域、准确读数与双向响应',
    f'完成 {N_FAMILIES} 个新家族；旧失败家族 {len(ANCHORS)} 个单独诊断。行为评分 {len(behavior_frame)}，干预 {len(patch_frame)}，逐层 trace {len(trace_frame)}。',
    '历史 Native/LoRA 重放、bare/disabled parity、零干预和实际剂量检查全部通过。方向完全复用旧冻结文件。',
    '## 四项冻结主检验',md_table(primary),
    'CI 为逐项 95% 区间；确认性判据使用四项统一 Holm p。侧交互定义为 (LoRA右−左)−(Native右−左)。投影坐标固定为 LoRA，注入轴固定为 Native L49，剂量 ±0.25。',
    '## 新几何的行为',md_table(behavior_summary[behavior_summary.scope.isin(['all','diagnostic_cell'])]),
    '准确读数只提供数量和距离。image_plus_facts 的改善不能独自证明视觉读取失败；facts_only 也改变模态和提示条件。各模型／形式的全部正确、错误及候选分数已保存。',
    '## 旧失败案例复测（不进入新主检验）',md_table(old_diagnostics),
    '## 左右侧的局部响应',md_table(slope_summary[(slope_summary.scope=='side')&(slope_summary.axis=='native')&(slope_summary.dose_magnitude==.25)&(slope_summary.response=='central_slope')]),
    '响应是 ±0.25 两次有限干预的差，不是自然 clean 信号量，也不是无穷小梯度。正负投影、输出 margin 和三分类正确性须分别解读。',
    '## 次级模型×侧交互',md_table(secondary),
    '这些次级区间未另作多重比较校正。随机轴提供同范数基线，单一随机方向不能穷尽所有控制方向。',
    '## 如何决定后续',
    '- 如果新邻域的图像错误被准确读数明显改善，再用独立读数任务及提示控制定位感知／信息形式因素。',
    '- 如果 L55 投影有限差分翻转而输出 margin 没有相应翻转，优先检查坐标与实际输出响应的关系；不能仅按该轴正负解释正确性。',
    '- 如果投影和输出响应共同呈现模型×侧差异，并在两个剂量及新中心子组重复，再设计逐输入匹配 L55 投影的验证；本轮没有完成该匹配。',
    '- 若本次主检验未通过，保留完整估计与控制，按这些固定邻域收窄结论，不据此重选剂量／家族。',
    '本研究只覆盖三个由旧失败选出的中心附近、一个 checkpoint/adapter/绘图样式。不同几何并不等同于整个任务的随机独立抽样；不声称唯一电路、算法替换、自然中介或失败由弱投影造成。',
    '## 文件',
    '`analysis/behavior_records.csv`、`intervention_records.csv` 保存三候选分数和正确性；`signed_state_traces.csv`、剂量表、逐输入有限差分、主检验家族值及全部新图已导出。']
if SOURCE_SYNTHETIC:lines.insert(1,'**SYNTHETIC ONLY：这是模拟模型验证，不是 27B 科研结果。**')
report='\n\n'.join(lines)+'\n';report_path=OUTPUT_ROOT/'LOCAL_RESPONSE_REPORT_ZH.md';report_path.write_text(report,encoding='utf-8')
write_json(OUTPUT_ROOT/'KEY_RESULTS.json',{'source_run':source_complete['run_identity'],'run_identity':RUN_DIGEST,
    'new_families':N_FAMILIES,'synthetic_only':SOURCE_SYNTHETIC,'primary':json.loads(primary.to_json(orient='records')),
    'behavior':json.loads(behavior_summary.to_json(orient='records')),'old_diagnostics':json.loads(old_diagnostics.to_json(orient='records')),
    'side_slopes':json.loads(slope_summary[slope_summary.scope=='side'].to_json(orient='records')),'controls':control})
for key,path in SOURCE_PATHS.items():require(sha256_file(path)==protocol['source_hashes'][key],'Source changed during run: '+key)
require(hash_tree(ADAPTER_DIR)==adapter_manifest['adapter_files'],'Adapter changed during run.')
require(hash_tree(DRIVE_TOOLKIT_ROOT/'src')==protocol['toolkit_files'],'Toolkit changed during run.')
require(sha256_file(axis_path)==AXIS_DIGEST,'Frozen axes changed during run.')
behavior_hashes={p.name:sha256_file(p) for p in sorted(BEHAVIOR_DIR.glob('*.json'))}
freeze_json(OUTPUT_ROOT/'behavior_shard_inventory.json',behavior_hashes)
outputs={p.relative_to(OUTPUT_ROOT).as_posix():sha256_file(p) for folder in (ANALYSIS_DIR,IMAGE_DIR) for p in sorted(folder.rglob('*')) if p.is_file()}
for p in [report_path,axis_path,*OUTPUT_ROOT.glob('*.json')]:
    if p.name!='RUN_COMPLETE.json':outputs[p.relative_to(OUTPUT_ROOT).as_posix()]=sha256_file(p)
write_json(OUTPUT_ROOT/'RUN_COMPLETE.json',{'status':'complete','run_identity':RUN_DIGEST,'outputs':outputs,
    'counts':{'behavior':len(behavior_frame),'interventions':len(patch_frame),'traces':len(trace_frame)},
    'n_families':N_FAMILIES,'old_diagnostic_families':len(ANCHORS),'synthetic_only':SOURCE_SYNTHETIC,
    'natural_mediation_proven':False,'l55_projection_matched':False,'completed_utc':datetime.now(timezone.utc).isoformat()})
with ZipFile(OUTPUT_ROOT/'review_bundle.zip','w',compression=8) as bundle:
    for name in [*outputs,'RUN_COMPLETE.json']:bundle.write(OUTPUT_ROOT/name,arcname=name)
display(Markdown(report));print('Report:',report_path);print('Review bundle:',OUTPUT_ROOT/'review_bundle.zip')
print('Finished. Save and return this executed notebook with its visible tables and figures.')